# Generate LaTeX Tables for Model Comparison

This notebook converts the saved outputs from `3Compare_model.ipynb` into publication-ready LaTeX tables.

The design is intentionally simple:

1. `3Compare_model.ipynb` computes and saves clean Parquet files.
2. This notebook reads one explicitly chosen result timestamp.
3. Every paper table has its own section, preview, caption, note, and `.tex` file.
4. The final section compiles all generated tables in one validation PDF.

The notebook does **not** rerun the forecast models. It only formats saved results. Raw stock-month and monthly panels are not exported as paper tables because they are source data, not reader-facing summaries.


## 1. Setup

Change only `TIMESTAMP` when a new model-comparison run becomes the paper version. Keeping the timestamp explicit makes every number in the paper reproducible.


In [1]:
from pathlib import Path
import shutil
import subprocess

import numpy as np
import pandas as pd
from scipy.stats import norm
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 100)

PROJECT_ROOT = Path("/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN")
TIMESTAMP = "20260831_1640"
RESULT_DIR = PROJECT_ROOT / "output/model_comparison"
TEX_DIR = PROJECT_ROOT / "overleaf/tab_tex"

MODEL_ORDER = [
    "rolling_normal",
    "linear_quantile",
    "fused_normal_k1",
    "fused_normal_k3",
    "structured_normal_k3",
]

MODEL_LABELS = {
    "rolling_normal": "Rolling Normal",
    "linear_quantile": "Linear Quantile",
    "fused_normal_k1": "Fused Normal K1",
    "fused_normal_k3": "Fused Normal K3",
    "structured_normal_k3": "Structured Normal K3",
}

METRIC_LABELS = {
    "nll": "NLL",
    "crps": "CRPS",
    "tail_crps_10": "Tail CRPS",
    "qs_1": "QS (1%)",
    "qs_5": "QS (5%)",
    "qs_10": "QS (10%)",
    "fz0_1": "FZ0 (1%)",
    "fz0_5": "FZ0 (5%)",
    "fz0_10": "FZ0 (10%)",
}

print("Project root:", PROJECT_ROOT)
print("Result timestamp:", TIMESTAMP)
print("LaTeX output folder:", TEX_DIR)


Project root: /Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN
Result timestamp: 20260831_1640
LaTeX output folder: /Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/overleaf/tab_tex


## 2. Load the saved results

Only compact result files needed for paper tables are loaded. Large stock-level component files are deliberately excluded.


## 3. Reusable LaTeX exporter

All tables use the same rules:

- one independent `.tex` file per table;
- English captions and notes for direct use in the paper;
- `booktabs`, `threeparttable`, and `adjustbox` formatting;
- `--` for missing values;
- a landscape wrapper only for genuinely wide tables.

The DataFrame shown in the notebook is the same formatted table written to LaTeX.


In [2]:
EXPORT_MANIFEST = []


def format_table(frame, formats=None, default_float=".4f"):
    """Return a string-valued table with stable publication formatting."""
    formats = formats or {}
    formatted = frame.copy()

    for column in formatted.columns:
        formatter = formats.get(column)

        def format_value(value):
            if pd.isna(value):
                return "--"
            if formatter is not None:
                if callable(formatter):
                    return str(formatter(value))
                return format(value, formatter)
            if isinstance(value, (float, np.floating)):
                return format(value, default_float)
            if isinstance(value, (int, np.integer)):
                return str(int(value))
            if isinstance(value, (pd.Timestamp, np.datetime64)):
                return pd.Timestamp(value).strftime("%Y-%m")
            return str(value)

        formatted[column] = formatted[column].map(format_value)

    return formatted


def export_latex_table(
    frame,
    filename,
    caption,
    label,
    note,
    formats=None,
    landscape=False,
    font_size="small",
):
    """Display one table and write one self-contained LaTeX table file."""
    formatted = format_table(frame, formats=formats)
    column_format = "l" + "r" * max(len(formatted.columns) - 1, 0)
    tabular = formatted.to_latex(
        index=False,
        escape=True,
        column_format=column_format,
    ).strip()

    table_position = "p" if landscape else "!htbp"
    table_text = f"""\\begin{{table}}[{table_position}]
                    \\centering
                    \\caption{{{caption}}}
                    \\label{{{label}}}
                    \\{font_size}
                    \\begin{{adjustbox}}{{max width=\\linewidth}}
                    {tabular}
                    \\end{{adjustbox}}
                    \\vspace{{0.25em}}

                    \\begin{{minipage}}{{0.98\\linewidth}}
                    \\footnotesize \\textit{{Notes:}} {note}
                    \\end{{minipage}}
                    \\end{{table}}
                    """
    if landscape:
        table_text = "\\begin{landscape}\n" + table_text + "\\end{landscape}\n"

    output_path = TEX_DIR / filename
    output_path.write_text(table_text, encoding="utf-8")

    EXPORT_MANIFEST.append(
        {
            "File": filename,
            "Caption": caption,
            "Label": label,
            "Rows": len(formatted),
            "Columns": len(formatted.columns),
            "Landscape": landscape,
        }
    )

    display(formatted)
    print("Saved:", output_path.relative_to(PROJECT_ROOT))
    return formatted


# Paper Tables

Each section below creates exactly one `.tex` file. The section text explains what the table is for and how to read it.


## Table 1. Common Sample Coverage

This table shows whether the five models are compared on the same stock-month sample. It belongs in the data or forecast-evaluation section.


In [3]:
table_name = "coverage_table"
path = RESULT_DIR / f"{table_name}_{TIMESTAMP}.parquet"
sample = pd.read_parquet(path)


table = pd.DataFrame({
    "Model": sample["model_id"].astype(object).map(MODEL_LABELS),
    "Available stock-months": sample["available_rows"],
    "Months": sample["months"],
    "First month": pd.to_datetime(sample["first_month"]),
    "Last month": pd.to_datetime(sample["last_month"]),
    "Common stock-months": sample["common_rows"],
    "Common share (%)": 100 * sample["common_share"],
})

export_latex_table(
    table,
    "coverage_table.tex",
    "Common Out-of-Sample Coverage",
    "tab:coverage_table",
    r"Available stock-months are observations produced by each model before the common-sample restriction. Common stock-months are observations shared by all five models. The common share is common stock-months divided by available stock-months.",
    formats={
        "Available stock-months": ",.0f",
        "Months": ".0f",
        "First month": lambda x: pd.Timestamp(x).strftime("%Y-%m"),
        "Last month": lambda x: pd.Timestamp(x).strftime("%Y-%m"),
        "Common stock-months": ",.0f",
        "Common share (%)": ".2f",
    },
)


,Model,Available stock-months,Months,First month,Last month,Common stock-months,Common share (%)
0,Rolling Normal,"480,446",216,2007-01,2024-12,"480,446",100.00
1,Linear Quantile,"480,446",216,2007-01,2024-12,"480,446",100.00
2,Fused Normal K1,"480,446",216,2007-01,2024-12,"480,446",100.00
3,Fused Normal K3,"480,446",216,2007-01,2024-12,"480,446",100.00
4,Structured Normal K3,"480,446",216,2007-01,2024-12,"480,446",100.00


Saved: overleaf/tab_tex/coverage_table.tex


,Model,Available stock-months,Months,First month,Last month,Common stock-months,Common share (%)
0,Rolling Normal,"480,446",216,2007-01,2024-12,"480,446",100.00
1,Linear Quantile,"480,446",216,2007-01,2024-12,"480,446",100.00
2,Fused Normal K1,"480,446",216,2007-01,2024-12,"480,446",100.00
3,Fused Normal K3,"480,446",216,2007-01,2024-12,"480,446",100.00
4,Structured Normal K3,"480,446",216,2007-01,2024-12,"480,446",100.00


## Table 2. FZ0 Domain Retention

The joint FZ0 score is defined only when its VaR and ES inputs satisfy the score's domain. This table reports how much of the sample remains valid.


In [4]:
table_name = "fz_domain_table"
path = RESULT_DIR / f"{table_name}_{TIMESTAMP}.parquet"
table = pd.read_parquet(path)
valid = table.pivot(index="model_id", columns="alpha", values="model_valid_share").reindex(MODEL_ORDER)
common = table.groupby("alpha")["all_model_common_share"].first()


table_02 = pd.DataFrame({
    "Model or sample": [MODEL_LABELS[m] for m in MODEL_ORDER] + ["All-model common sample"],
    "1% retained": list(100 * valid[0.01]) + [100 * common.loc[0.01]],
    "5% retained": list(100 * valid[0.05]) + [100 * common.loc[0.05]],
    "10% retained": list(100 * valid[0.10]) + [100 * common.loc[0.10]],
})

export_latex_table(
    table_02,
    "fz_domain_table.tex",
    "FZ0 Domain Retention Rates",
    "tab:fz_domain",
    r"Entries are percentages of the common forecast sample. Model rows report the share satisfying that model's FZ0 domain restrictions. The final row reports the share that is jointly valid for all five models and is therefore used in model comparisons.",
    formats={"1% retained": ".2f", "5% retained": ".2f", "10% retained": ".2f"},
)


,Model or sample,1% retained,5% retained,10% retained
0,Rolling Normal,99.83,99.74,99.63
1,Linear Quantile,99.70,98.82,97.44
2,Fused Normal K1,100.00,100.00,99.87
3,Fused Normal K3,100.00,99.93,99.18
4,Structured Normal K3,100.00,100.00,100.00
5,All-model common sample,99.52,98.49,96.28


Saved: overleaf/tab_tex/fz_domain_table.tex


,Model or sample,1% retained,5% retained,10% retained
0,Rolling Normal,99.83,99.74,99.63
1,Linear Quantile,99.70,98.82,97.44
2,Fused Normal K1,100.00,100.00,99.87
3,Fused Normal K3,100.00,99.93,99.18
4,Structured Normal K3,100.00,100.00,100.00
5,All-model common sample,99.52,98.49,96.28


## Table 3. Primary Forecast Performance

This is the main forecast-comparison table. It contains the two pre-specified primary metrics and their ranks.


In [5]:
table_name = "comparison_summary"
path = RESULT_DIR / f"{table_name}_{TIMESTAMP}.parquet"
summary = pd.read_parquet(path).set_index("model_id").reindex(MODEL_ORDER)

table_03 = pd.DataFrame({
    "Model": [MODEL_LABELS[m] for m in MODEL_ORDER],
    "FZ0 (5%)": summary["fz0_5"].to_numpy(),
    # "FZ0 months": summary["fz0_months_5"].to_numpy(),
    "FZ0 rank": summary["rank_fz0_5"].to_numpy(),
    "Tail CRPS": summary["tail_crps_10"].to_numpy(),
    "Tail CRPS rank": summary["rank_tail_crps_10"].to_numpy(),
})

export_latex_table(
    table_03,
    "03_primary_forecast_performance.tex",
    "Primary Out-of-Sample Forecast Performance",
    "tab:primary_forecast_performance",
    r"The table gives equal-month averages over the common stock-month sample. FZ0 jointly evaluates 5\% VaR and ES. Tail CRPS evaluates the 0--10\% lower tail. Lower scores and lower ranks indicate better forecasts. FZ0 months is the number of monthly observations entering the FZ0 average.",
    formats={
        "FZ0 (5%)": ".6f",
        # "FZ0 months": ".0f",
        "FZ0 rank": ".0f",
        "Tail CRPS": ".6f",
        "Tail CRPS rank": ".0f",
    },
)


,Model,FZ0 (5%),FZ0 rank,Tail CRPS,Tail CRPS rank
0,Rolling Normal,-1.123400,3,0.029370,3
1,Linear Quantile,-0.101718,5,0.029658,5
2,Fused Normal K1,-1.194602,2,0.028619,2
3,Fused Normal K3,-1.107867,4,0.029464,4
4,Structured Normal K3,-1.249098,1,0.027792,1


Saved: overleaf/tab_tex/03_primary_forecast_performance.tex


,Model,FZ0 (5%),FZ0 rank,Tail CRPS,Tail CRPS rank
0,Rolling Normal,-1.123400,3,0.029370,3
1,Linear Quantile,-0.101718,5,0.029658,5
2,Fused Normal K1,-1.194602,2,0.028619,2
3,Fused Normal K3,-1.107867,4,0.029464,4
4,Structured Normal K3,-1.249098,1,0.027792,1


## Table 4. Secondary Forecast Scores

This wide appendix table reports every secondary score and its cross-model rank.


In [6]:
secondary_metrics = ["nll", "crps", "qs_1", "qs_5", "qs_10", "fz0_1", "fz0_10"]
table_04 = pd.DataFrame({"Model": [MODEL_LABELS[m] for m in MODEL_ORDER]})
formats_04 = {}

for metric in secondary_metrics:
    label = METRIC_LABELS[metric]
    table_04[label] = summary[metric].to_numpy()
    # table_04[f"{label} rank"] = summary[f"rank_{metric}"].to_numpy()
    formats_04[label] = ".3f" if metric == "nll" else ".6f"
    # formats_04[f"{label} rank"] = ".0f"

export_latex_table(
    table_04,
    "04_secondary_forecast_scores.tex",
    "Secondary Out-of-Sample Forecast Scores",
    "tab:secondary_forecast_scores",
    r"The table gives equal-month averages over the common stock-month sample. Lower scores and lower ranks indicate better forecasts. NLL is unavailable for Linear Quantile because that model does not produce an exact continuous density.",
    formats=formats_04,
    landscape=True,
    font_size="scriptsize",
)


,Model,NLL,CRPS,QS (1%),QS (5%),QS (10%),FZ0 (1%),FZ0 (10%)
0,Rolling Normal,61486.849,0.073582,0.010932,0.031244,0.048734,-0.016419,-1.281620
1,Linear Quantile,--,0.078339,0.009465,0.031586,0.051127,-0.409896,-0.600207
2,Fused Normal K1,-0.477,0.076073,0.008981,0.030376,0.049702,-0.720875,-1.402141
3,Fused Normal K3,-0.531,0.077778,0.009029,0.031490,0.050951,-0.775057,-1.280554
4,Structured Normal K3,-0.624,0.072622,0.008743,0.029619,0.047983,-0.808196,-1.477641


Saved: overleaf/tab_tex/04_secondary_forecast_scores.tex


,Model,NLL,CRPS,QS (1%),QS (5%),QS (10%),FZ0 (1%),FZ0 (10%)
0,Rolling Normal,61486.849,0.073582,0.010932,0.031244,0.048734,-0.016419,-1.281620
1,Linear Quantile,--,0.078339,0.009465,0.031586,0.051127,-0.409896,-0.600207
2,Fused Normal K1,-0.477,0.076073,0.008981,0.030376,0.049702,-0.720875,-1.402141
3,Fused Normal K3,-0.531,0.077778,0.009029,0.031490,0.050951,-0.775057,-1.280554
4,Structured Normal K3,-0.624,0.072622,0.008743,0.029619,0.047983,-0.808196,-1.477641


## Table 5. VaR Coverage Summary

Coverage should be close to its target rate. The rank is based on the absolute coverage error, not on the coverage rate itself.


In [7]:
# table_name = "comparison_summary"
# path = RESULT_DIR / f"{table_name}_{TIMESTAMP}.parquet"
# summary = pd.read_parquet(path).set_index("model_id").reindex(MODEL_ORDER)


table_05 = pd.DataFrame({"Model": [MODEL_LABELS[m] for m in MODEL_ORDER]})
formats_05 = {}

for suffix, target in [("1", 1), ("5", 5), ("10", 10)]:
    table_05[f"Coverage {target}%"] = 100 * summary[f"coverage_{suffix}"].to_numpy()
    table_05[f"Error {target} (pp)"] = 100 * summary[f"coverage_error_{suffix}"].to_numpy()
    table_05[f"Rank {target}%"] = summary[f"rank_abs_coverage_error_{suffix}"].to_numpy()
    formats_05[f"Coverage {target}%"] = ".3f"
    formats_05[f"Error {target} (pp)"] = ".3f"
    formats_05[f"Rank {target}%"] = ".0f"

export_latex_table(
    table_05,
    "05_var_coverage_summary.tex",
    "Value-at-Risk Coverage Summary",
    "tab:var_coverage_summary",
    r"Coverage is the percentage of realized returns below the predicted VaR. Error is coverage minus the nominal target, in percentage points. The rank is based on the absolute value of the error, so a rank of one is the closest to correct unconditional coverage.",
    formats=formats_05,
    landscape=True,
    font_size="scriptsize",
)


,Model,Coverage 1%,Error 1 (pp),Rank 1%,Coverage 5%,Error 5 (pp),Rank 5%,Coverage 10%,Error 10 (pp),Rank 10%
0,Rolling Normal,2.289,1.289,5,5.858,0.858,3,10.130,0.130,2
1,Linear Quantile,1.529,0.529,1,5.701,0.701,2,9.965,-0.035,1
2,Fused Normal K1,1.942,0.942,4,5.176,0.176,1,8.675,-1.325,5
3,Fused Normal K3,1.765,0.765,2,6.657,1.657,5,11.036,1.036,4
4,Structured Normal K3,1.929,0.929,3,6.362,1.362,4,10.872,0.872,3


Saved: overleaf/tab_tex/05_var_coverage_summary.tex


,Model,Coverage 1%,Error 1 (pp),Rank 1%,Coverage 5%,Error 5 (pp),Rank 5%,Coverage 10%,Error 10 (pp),Rank 10%
0,Rolling Normal,2.289,1.289,5,5.858,0.858,3,10.130,0.130,2
1,Linear Quantile,1.529,0.529,1,5.701,0.701,2,9.965,-0.035,1
2,Fused Normal K1,1.942,0.942,4,5.176,0.176,1,8.675,-1.325,5
3,Fused Normal K3,1.765,0.765,2,6.657,1.657,5,11.036,1.036,4
4,Structured Normal K3,1.929,0.929,3,6.362,1.362,4,10.872,0.872,3


## Table 6. Primary Pairwise Forecast Tests

This table implements the pre-specified model ladder for the two primary loss functions.


In [8]:
table_name = "pairwise_tests"
path = RESULT_DIR / f"{table_name}_{TIMESTAMP}.parquet"
pairwise = pd.read_parquet(path)


STEP_LABELS = {
    "条件信息": "Conditional information",
    "非线性单分布": "Nonlinear single density",
    "Mixture": "Flexible mixture",
    "结构化参数映射": "Structured mapping",
}


def prepare_pairwise_table(frame):
    ordered = frame.copy()
    ordered["candidate"] = pd.Categorical(ordered["candidate"], MODEL_ORDER, ordered=True)
    ordered = ordered.sort_values(["candidate", "metric"])
    return pd.DataFrame({
        "Comparison": [
            f"{MODEL_LABELS[str(c)]} vs. {MODEL_LABELS[b]}"
            for c, b in zip(ordered["candidate"], ordered["benchmark"])
        ],
        "Model step": ordered["comparison"].map(STEP_LABELS),
        "Metric": ordered["metric"].map(METRIC_LABELS),
        "Mean difference": ordered["mean_difference"],
        "NW SE": ordered["nw_se"],
        "NW t": ordered["nw_t"],
        "p-value": ordered["p_two_sided"],
        "Months": ordered["months"],
        "Lower-loss model": ordered["lower_mean_loss"].map(MODEL_LABELS),
    })


primary_pairwise = pairwise[pairwise["metric"].isin(["fz0_5", "tail_crps_10"])]
table_06 = prepare_pairwise_table(primary_pairwise)

export_latex_table(
    table_06,
    "06_primary_pairwise_tests.tex",
    "Primary Pairwise Forecast Comparisons",
    "tab:primary_pairwise_tests",
    r"Mean difference is the candidate model's monthly loss minus the benchmark model's monthly loss. A negative value favors the candidate. NW SE and NW t use Newey--West standard errors with 12 lags. The comparisons follow the pre-specified model ladder.",
    formats={
        "Mean difference": ".6f",
        "NW SE": ".6f",
        "NW t": ".3f",
        "p-value": ".4f",
        "Months": ".0f",
    },
    landscape=True,
    font_size="scriptsize",
)


,Comparison,Model step,Metric,Mean difference,NW SE,NW t,p-value,Months,Lower-loss model
0,Linear Quantile vs. Rolling Normal,Conditional information,FZ0 (5%),1.021682,0.820274,1.246,0.2129,216,Rolling Normal
1,Linear Quantile vs. Rolling Normal,Conditional information,Tail CRPS,0.000288,0.001221,0.236,0.8137,216,Rolling Normal
6,Fused Normal K1 vs. Linear Quantile,Nonlinear single density,FZ0 (5%),-1.092884,0.810322,-1.349,0.1774,216,Fused Normal K1
7,Fused Normal K1 vs. Linear Quantile,Nonlinear single density,Tail CRPS,-0.001039,0.000963,-1.079,0.2807,216,Fused Normal K1
12,Fused Normal K3 vs. Fused Normal K1,Flexible mixture,FZ0 (5%),0.086735,0.045784,1.894,0.0582,216,Fused Normal K1
13,Fused Normal K3 vs. Fused Normal K1,Flexible mixture,Tail CRPS,0.000846,0.000504,1.677,0.0936,216,Fused Normal K1
19,Structured Normal K3 vs. Fused Normal K3,Structured mapping,FZ0 (5%),-0.141231,0.070894,-1.992,0.0464,216,Structured Normal K3
20,Structured Normal K3 vs. Fused Normal K3,Structured mapping,Tail CRPS,-0.001673,0.000723,-2.315,0.0206,216,Structured Normal K3


Saved: overleaf/tab_tex/06_primary_pairwise_tests.tex


,Comparison,Model step,Metric,Mean difference,NW SE,NW t,p-value,Months,Lower-loss model
0,Linear Quantile vs. Rolling Normal,Conditional information,FZ0 (5%),1.021682,0.820274,1.246,0.2129,216,Rolling Normal
1,Linear Quantile vs. Rolling Normal,Conditional information,Tail CRPS,0.000288,0.001221,0.236,0.8137,216,Rolling Normal
6,Fused Normal K1 vs. Linear Quantile,Nonlinear single density,FZ0 (5%),-1.092884,0.810322,-1.349,0.1774,216,Fused Normal K1
7,Fused Normal K1 vs. Linear Quantile,Nonlinear single density,Tail CRPS,-0.001039,0.000963,-1.079,0.2807,216,Fused Normal K1
12,Fused Normal K3 vs. Fused Normal K1,Flexible mixture,FZ0 (5%),0.086735,0.045784,1.894,0.0582,216,Fused Normal K1
13,Fused Normal K3 vs. Fused Normal K1,Flexible mixture,Tail CRPS,0.000846,0.000504,1.677,0.0936,216,Fused Normal K1
19,Structured Normal K3 vs. Fused Normal K3,Structured mapping,FZ0 (5%),-0.141231,0.070894,-1.992,0.0464,216,Structured Normal K3
20,Structured Normal K3 vs. Fused Normal K3,Structured mapping,Tail CRPS,-0.001673,0.000723,-2.315,0.0206,216,Structured Normal K3


## Table 7. Secondary Pairwise Forecast Tests

This appendix table applies the same model ladder to all remaining forecast scores.


In [9]:


secondary_pairwise = pairwise[~pairwise["metric"].isin(["fz0_5", "tail_crps_10"])]
table_07 = prepare_pairwise_table(secondary_pairwise)

export_latex_table(
    table_07,
    "07_secondary_pairwise_tests.tex",
    "Secondary Pairwise Forecast Comparisons",
    "tab:secondary_pairwise_tests",
    r"Mean difference is the candidate model's monthly loss minus the benchmark model's monthly loss. A negative value favors the candidate. NW SE and NW t use Newey--West standard errors with 12 lags. NLL comparisons exclude Linear Quantile because it does not produce an exact continuous density.",
    formats={
        "Mean difference": ".6f",
        "NW SE": ".6f",
        "NW t": ".3f",
        "p-value": ".4f",
        "Months": ".0f",
    },
    landscape=True,
    font_size="scriptsize",
)


,Comparison,Model step,Metric,Mean difference,NW SE,NW t,p-value,Months,Lower-loss model
2,Linear Quantile vs. Rolling Normal,Conditional information,CRPS,0.004757,0.004271,1.114,0.2654,216,Rolling Normal
3,Linear Quantile vs. Rolling Normal,Conditional information,QS (1%),-0.001467,0.000825,-1.778,0.0755,216,Linear Quantile
5,Linear Quantile vs. Rolling Normal,Conditional information,QS (10%),0.002393,0.001779,1.345,0.1786,216,Rolling Normal
4,Linear Quantile vs. Rolling Normal,Conditional information,QS (5%),0.000341,0.001336,0.255,0.7985,216,Rolling Normal
8,Fused Normal K1 vs. Linear Quantile,Nonlinear single density,CRPS,-0.002266,0.004678,-0.484,0.6281,216,Fused Normal K1
9,Fused Normal K1 vs. Linear Quantile,Nonlinear single density,QS (1%),-0.000484,0.000583,-0.830,0.4067,216,Fused Normal K1
11,Fused Normal K1 vs. Linear Quantile,Nonlinear single density,QS (10%),-0.001425,0.001663,-0.857,0.3913,216,Fused Normal K1
10,Fused Normal K1 vs. Linear Quantile,Nonlinear single density,QS (5%),-0.001210,0.001009,-1.199,0.2305,216,Fused Normal K1
14,Fused Normal K3 vs. Fused Normal K1,Flexible mixture,CRPS,0.001706,0.001080,1.579,0.1142,216,Fused Normal K1
18,Fused Normal K3 vs. Fused Normal K1,Flexible mixture,NLL,-0.054743,0.030296,-1.807,0.0708,216,Fused Normal K3


Saved: overleaf/tab_tex/07_secondary_pairwise_tests.tex


,Comparison,Model step,Metric,Mean difference,NW SE,NW t,p-value,Months,Lower-loss model
2,Linear Quantile vs. Rolling Normal,Conditional information,CRPS,0.004757,0.004271,1.114,0.2654,216,Rolling Normal
3,Linear Quantile vs. Rolling Normal,Conditional information,QS (1%),-0.001467,0.000825,-1.778,0.0755,216,Linear Quantile
5,Linear Quantile vs. Rolling Normal,Conditional information,QS (10%),0.002393,0.001779,1.345,0.1786,216,Rolling Normal
4,Linear Quantile vs. Rolling Normal,Conditional information,QS (5%),0.000341,0.001336,0.255,0.7985,216,Rolling Normal
8,Fused Normal K1 vs. Linear Quantile,Nonlinear single density,CRPS,-0.002266,0.004678,-0.484,0.6281,216,Fused Normal K1
9,Fused Normal K1 vs. Linear Quantile,Nonlinear single density,QS (1%),-0.000484,0.000583,-0.830,0.4067,216,Fused Normal K1
11,Fused Normal K1 vs. Linear Quantile,Nonlinear single density,QS (10%),-0.001425,0.001663,-0.857,0.3913,216,Fused Normal K1
10,Fused Normal K1 vs. Linear Quantile,Nonlinear single density,QS (5%),-0.001210,0.001009,-1.199,0.2305,216,Fused Normal K1
14,Fused Normal K3 vs. Fused Normal K1,Flexible mixture,CRPS,0.001706,0.001080,1.579,0.1142,216,Fused Normal K1
18,Fused Normal K3 vs. Fused Normal K1,Flexible mixture,NLL,-0.054743,0.030296,-1.807,0.0708,216,Fused Normal K3


## Table 8. PIT Calibration

This table summarizes whether each model's full distribution is probabilistically calibrated.


In [10]:
table_name = "pit_summary"
path = RESULT_DIR / f"{table_name}_{TIMESTAMP}.parquet"
pit = pd.read_parquet(path).set_index("model_id").reindex(MODEL_ORDER)

table_08 = pd.DataFrame({
    "Model": [MODEL_LABELS[m] for m in MODEL_ORDER],
    "PIT mean": pit["pit_mean"].to_numpy(),
    "Mean error": pit["pit_mean_error"].to_numpy(),
    "PIT variance": pit["pit_variance"].to_numpy(),
    "Variance error": pit["pit_variance_error"].to_numpy(),
    "KS distance": pit["pit_ks"].to_numpy(),
    "Monthly-mean ACF(1)": pit["pit_mean_acf1"].to_numpy(),
})

export_latex_table(
    table_08,
    "08_pit_summary.tex",
    "Probability Integral Transform Diagnostics",
    "tab:pit_summary",
    r"For a calibrated continuous density, PIT values are uniform on zero to one, with mean 0.5 and variance 1/12. Mean and variance errors subtract these targets. Smaller KS distance and smaller absolute first-order autocorrelation indicate better calibration. Linear Quantile PIT values use interpolation across its predicted quantile grid.",
    formats={column: ".6f" for column in table_08.columns if column != "Model"},
)


,Model,PIT mean,Mean error,PIT variance,Variance error,KS distance,Monthly-mean ACF(1)
0,Rolling Normal,0.476341,-0.023659,0.061390,-0.021944,0.196653,-0.054371
1,Linear Quantile,0.536981,0.036981,0.059942,-0.023391,0.250505,0.079587
2,Fused Normal K1,0.552531,0.052531,0.059890,-0.023444,0.243927,0.179132
3,Fused Normal K3,0.557293,0.057293,0.064953,-0.018380,0.260534,0.251569
4,Structured Normal K3,0.527892,0.027892,0.066141,-0.017192,0.222096,0.018201


Saved: overleaf/tab_tex/08_pit_summary.tex


,Model,PIT mean,Mean error,PIT variance,Variance error,KS distance,Monthly-mean ACF(1)
0,Rolling Normal,0.476341,-0.023659,0.061390,-0.021944,0.196653,-0.054371
1,Linear Quantile,0.536981,0.036981,0.059942,-0.023391,0.250505,0.079587
2,Fused Normal K1,0.552531,0.052531,0.059890,-0.023444,0.243927,0.179132
3,Fused Normal K3,0.557293,0.057293,0.064953,-0.018380,0.260534,0.251569
4,Structured Normal K3,0.527892,0.027892,0.066141,-0.017192,0.222096,0.018201


## Table 9. VaR Coverage Tests

This table adds formal Newey--West tests of whether observed VaR coverage equals the nominal target.


In [11]:
table_name = "coverage_tests"
path = RESULT_DIR / f"{table_name}_{TIMESTAMP}.parquet"
coverage = pd.read_parquet(path)

coverage["model_id"] = pd.Categorical(coverage["model_id"], MODEL_ORDER, ordered=True)
coverage = coverage.sort_values(["model_id", "alpha"])

table_09 = pd.DataFrame({
    "Model": coverage["model_id"].astype(object).map(MODEL_LABELS),
    "Target": 100 * coverage["alpha"],
    "Coverage (%)": 100 * coverage["coverage_rate"],
    "Error (pp)": 100 * coverage["coverage_error"],
    "NW SE (pp)": 100 * coverage["nw_se"],
    "NW t": coverage["nw_t"],
    "p-value": coverage["p_two_sided"],
    "Months": coverage["months"],
})

export_latex_table(
    table_09,
    "09_var_coverage_tests.tex",
    "Value-at-Risk Coverage Tests",
    "tab:var_coverage_tests",
    r"The null hypothesis is that observed coverage equals the nominal VaR target. Error and its Newey--West standard error are reported in percentage points. The test uses 12 lags. A positive error means that losses cross the VaR threshold too often.",
    formats={
        "Target": ".0f",
        "Coverage (%)": ".3f",
        "Error (pp)": ".3f",
        "NW SE (pp)": ".3f",
        "NW t": ".3f",
        "p-value": ".4f",
        "Months": ".0f",
    },
    landscape=True,
    font_size="scriptsize",
)


,Model,Target,Coverage (%),Error (pp),NW SE (pp),NW t,p-value,Months
0,Rolling Normal,1,2.289,1.289,0.476,2.706,0.0068,216
1,Rolling Normal,5,5.858,0.858,0.849,1.011,0.3119,216
2,Rolling Normal,10,10.130,0.130,1.098,0.119,0.9055,216
3,Linear Quantile,1,1.529,0.529,0.485,1.091,0.2752,216
4,Linear Quantile,5,5.701,0.701,1.332,0.527,0.5984,216
5,Linear Quantile,10,9.965,-0.035,1.946,-0.018,0.9858,216
6,Fused Normal K1,1,1.942,0.942,0.447,2.107,0.0351,216
7,Fused Normal K1,5,5.176,0.176,0.981,0.179,0.8577,216
8,Fused Normal K1,10,8.675,-1.325,1.427,-0.928,0.3532,216
9,Fused Normal K3,1,1.765,0.765,0.462,1.656,0.0977,216


Saved: overleaf/tab_tex/09_var_coverage_tests.tex


,Model,Target,Coverage (%),Error (pp),NW SE (pp),NW t,p-value,Months
0,Rolling Normal,1,2.289,1.289,0.476,2.706,0.0068,216
1,Rolling Normal,5,5.858,0.858,0.849,1.011,0.3119,216
2,Rolling Normal,10,10.130,0.130,1.098,0.119,0.9055,216
3,Linear Quantile,1,1.529,0.529,0.485,1.091,0.2752,216
4,Linear Quantile,5,5.701,0.701,1.332,0.527,0.5984,216
5,Linear Quantile,10,9.965,-0.035,1.946,-0.018,0.9858,216
6,Fused Normal K1,1,1.942,0.942,0.447,2.107,0.0351,216
7,Fused Normal K1,5,5.176,0.176,0.981,0.179,0.8577,216
8,Fused Normal K1,10,8.675,-1.325,1.427,-0.928,0.3532,216
9,Fused Normal K3,1,1.765,0.765,0.462,1.656,0.0977,216


## Table 10. Quantile Consistency Checks

This numerical check confirms that the VaR values used in the joint score agree with the corresponding points on the common quantile grid.


In [12]:
table_name = "quantile_check_table"
path = RESULT_DIR / f"{table_name}_{TIMESTAMP}.parquet"
checks = pd.read_parquet(path).set_index("model_id").reindex(MODEL_ORDER)

table_10 = pd.DataFrame({
    "Model": [MODEL_LABELS[m] for m in MODEL_ORDER],
    "Maximum difference at 1%": checks["var_1_max_abs_diff"].to_numpy(),
    "Maximum difference at 5%": checks["var_5_max_abs_diff"].to_numpy(),
    "Maximum difference at 10%": checks["var_10_max_abs_diff"].to_numpy(),
})

export_latex_table(
    table_10,
    "10_quantile_consistency_checks.tex",
    "Quantile and Value-at-Risk Consistency Checks",
    "tab:quantile_consistency_checks",
    r"Each entry is the maximum absolute difference between the reported VaR and the matching point on the shared quantile grid over the out-of-sample observations. Values close to zero confirm internal consistency.",
    formats={column: ".2e" for column in table_10.columns if column != "Model"},
)


,Model,Maximum difference at 1%,Maximum difference at 5%,Maximum difference at 10%
0,Rolling Normal,0.00e+00,0.00e+00,0.00e+00
1,Linear Quantile,0.00e+00,0.00e+00,0.00e+00
2,Fused Normal K1,2.76e-13,1.29e-12,5.83e-13
3,Fused Normal K3,3.18e-12,3.17e-12,3.16e-12
4,Structured Normal K3,2.29e-12,2.15e-12,2.31e-12


Saved: overleaf/tab_tex/10_quantile_consistency_checks.tex


,Model,Maximum difference at 1%,Maximum difference at 5%,Maximum difference at 10%
0,Rolling Normal,0.00e+00,0.00e+00,0.00e+00
1,Linear Quantile,0.00e+00,0.00e+00,0.00e+00
2,Fused Normal K1,2.76e-13,1.29e-12,5.83e-13
3,Fused Normal K3,3.18e-12,3.17e-12,3.16e-12
4,Structured Normal K3,2.29e-12,2.15e-12,2.31e-12


## Table 11. Component Labels by Forecast Window

The Structured K3 network does not give its three raw components permanent economic names. This table records how the adverse, normal, and favorable labels are assigned in each rolling window.


In [13]:
table_name = "comparison_summary"
path = RESULT_DIR / f"{table_name}_{TIMESTAMP}.parquet"
labels = pd.read_parquet(path)


# labels = results["component_label_map"].copy()

table_11 = labels.rename(columns={
    "window": "Window",
    "test_period": "Test period",
    "adverse_component": "Adverse ID",
    "normal_component": "Normal ID",
    "favorable_component": "Favorable ID",
    "trainval_mean_mu_adverse": "Adverse mean",
    "trainval_mean_mu_normal": "Normal mean",
    "trainval_mean_mu_favorable": "Favorable mean",
})

export_latex_table(
    table_11,
    "11_component_label_map.tex",
    "Structured K3 Component Labels by Forecast Window",
    "tab:component_label_map",
    r"Component IDs are zero-based raw network outputs. Within each rolling window, labels are assigned from month-equal component means in the combined Train and Validation samples. The lowest-mean component is adverse, the middle component is normal, and the highest-mean component is favorable. The mapping is frozen before the Test year.",
    formats={
        "Window": ".0f",
        "Adverse ID": ".0f",
        "Normal ID": ".0f",
        "Favorable ID": ".0f",
        "Adverse mean": ".6f",
        "Normal mean": ".6f",
        "Favorable mean": ".6f",
    },
    landscape=True,
    font_size="scriptsize",
)


,model_id,months,mean_firms_per_month,nll,crps,tail_crps_10,qs_1,qs_5,qs_10,fz0_1,fz0_5,fz0_10,coverage_1,coverage_5,coverage_10,coverage_error_1,fz0_months_1,coverage_error_5,fz0_months_5,coverage_error_10,fz0_months_10,rank_fz0_5,rank_tail_crps_10,rank_nll,rank_crps,rank_qs_1,rank_qs_5,rank_qs_10,rank_fz0_1,rank_fz0_10,rank_abs_coverage_error_1,rank_abs_coverage_error_5,rank_abs_coverage_error_10
0,rolling_normal,216,2224.2870,61486.8486,0.0736,0.0294,0.0109,0.0312,0.0487,-0.0164,-1.1234,-1.2816,0.0229,0.0586,0.1013,0.0129,216,0.0086,216,0.0013,216,3.0000,3.0000,4.0000,2.0000,5.0000,3.0000,2.0000,5.0000,3.0000,5.0000,3.0000,2.0000
1,linear_quantile,216,2224.2870,--,0.0783,0.0297,0.0095,0.0316,0.0511,-0.4099,-0.1017,-0.6002,0.0153,0.0570,0.0997,0.0053,216,0.0070,216,-0.0003,216,5.0000,5.0000,--,5.0000,4.0000,5.0000,5.0000,4.0000,5.0000,1.0000,2.0000,1.0000
2,fused_normal_k1,216,2224.2870,-0.4765,0.0761,0.0286,0.0090,0.0304,0.0497,-0.7209,-1.1946,-1.4021,0.0194,0.0518,0.0867,0.0094,216,0.0018,216,-0.0133,216,2.0000,2.0000,3.0000,3.0000,2.0000,2.0000,3.0000,3.0000,2.0000,4.0000,1.0000,5.0000
3,fused_normal_k3,216,2224.2870,-0.5313,0.0778,0.0295,0.0090,0.0315,0.0510,-0.7751,-1.1079,-1.2806,0.0177,0.0666,0.1104,0.0077,216,0.0166,216,0.0104,216,4.0000,4.0000,2.0000,4.0000,3.0000,4.0000,4.0000,2.0000,4.0000,2.0000,5.0000,4.0000
4,structured_normal_k3,216,2224.2870,-0.6243,0.0726,0.0278,0.0087,0.0296,0.0480,-0.8082,-1.2491,-1.4776,0.0193,0.0636,0.1087,0.0093,216,0.0136,216,0.0087,216,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,3.0000,4.0000,3.0000


Saved: overleaf/tab_tex/11_component_label_map.tex


,model_id,months,mean_firms_per_month,nll,crps,tail_crps_10,qs_1,qs_5,qs_10,fz0_1,fz0_5,fz0_10,coverage_1,coverage_5,coverage_10,coverage_error_1,fz0_months_1,coverage_error_5,fz0_months_5,coverage_error_10,fz0_months_10,rank_fz0_5,rank_tail_crps_10,rank_nll,rank_crps,rank_qs_1,rank_qs_5,rank_qs_10,rank_fz0_1,rank_fz0_10,rank_abs_coverage_error_1,rank_abs_coverage_error_5,rank_abs_coverage_error_10
0,rolling_normal,216,2224.2870,61486.8486,0.0736,0.0294,0.0109,0.0312,0.0487,-0.0164,-1.1234,-1.2816,0.0229,0.0586,0.1013,0.0129,216,0.0086,216,0.0013,216,3.0000,3.0000,4.0000,2.0000,5.0000,3.0000,2.0000,5.0000,3.0000,5.0000,3.0000,2.0000
1,linear_quantile,216,2224.2870,--,0.0783,0.0297,0.0095,0.0316,0.0511,-0.4099,-0.1017,-0.6002,0.0153,0.0570,0.0997,0.0053,216,0.0070,216,-0.0003,216,5.0000,5.0000,--,5.0000,4.0000,5.0000,5.0000,4.0000,5.0000,1.0000,2.0000,1.0000
2,fused_normal_k1,216,2224.2870,-0.4765,0.0761,0.0286,0.0090,0.0304,0.0497,-0.7209,-1.1946,-1.4021,0.0194,0.0518,0.0867,0.0094,216,0.0018,216,-0.0133,216,2.0000,2.0000,3.0000,3.0000,2.0000,2.0000,3.0000,3.0000,2.0000,4.0000,1.0000,5.0000
3,fused_normal_k3,216,2224.2870,-0.5313,0.0778,0.0295,0.0090,0.0315,0.0510,-0.7751,-1.1079,-1.2806,0.0177,0.0666,0.1104,0.0077,216,0.0166,216,0.0104,216,4.0000,4.0000,2.0000,4.0000,3.0000,4.0000,4.0000,2.0000,4.0000,2.0000,5.0000,4.0000
4,structured_normal_k3,216,2224.2870,-0.6243,0.0726,0.0278,0.0087,0.0296,0.0480,-0.8082,-1.2491,-1.4776,0.0193,0.0636,0.1087,0.0093,216,0.0136,216,0.0087,216,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,3.0000,4.0000,3.0000


## Table 12. Component Numerical Diagnostics

This compact table checks whether the component decomposition is mechanically correct and economically non-degenerate.


In [14]:
table_name = "component_check_table"
path = RESULT_DIR / f"{table_name}_{TIMESTAMP}.parquet"
component_checks = pd.read_parquet(path).set_index("check")["value"]

check_specs = [
    ("max_ES_reconstruction_error", "Maximum ES reconstruction error", ".2e", "Should be close to zero"),
    ("max_tail_weight_sum_error", "Maximum tail-weight sum error", ".2e", "Should be close to zero"),
    ("max_within_month_pi_spread", "Maximum within-month weight spread", ".2e", "Should be zero under common monthly weights"),
    ("mean_effective_k", "Mean effective component count", ".4f", "Above one means more than one component is used"),
    ("collapse_month_share", "Collapsed-month share", ".2%", "Share of months dominated by one component"),
    ("adverse_lowest_month_share", "Adverse-lowest month share", ".2%", "Share of months with the adverse mean lowest"),
    ("adverse_lowest_firm_share", "Adverse-lowest firm share", ".2%", "Share of stock-months with the adverse mean lowest"),
    ("adverse_ES_contribution_ratio", "Adverse ES contribution ratio", ".2%", "Adverse contribution divided by total ES"),
]

table_12 = pd.DataFrame([
    {
        "Diagnostic": label,
        "Value": format(component_checks.loc[key], value_format),
        "Interpretation": interpretation,
    }
    for key, label, value_format, interpretation in check_specs
])

export_latex_table(
    table_12,
    "12_component_numerical_checks.tex",
    "Structured K3 Component Diagnostics",
    "tab:component_numerical_checks",
    r"The first three rows test numerical identities. The remaining rows test whether the mixture uses multiple components and whether the component labeled adverse remains economically meaningful in the Test sample.",
)


,Diagnostic,Value,Interpretation
0,Maximum ES reconstruction error,4.44e-16,Should be close to zero
1,Maximum tail-weight sum error,6.73e-13,Should be close to zero
2,Maximum within-month weight spread,0.00e+00,Should be zero under common monthly weights
3,Mean effective component count,1.7962,Above one means more than one component is used
4,Collapsed-month share,0.93%,Share of months dominated by one component
5,Adverse-lowest month share,100.00%,Share of months with the adverse mean lowest
6,Adverse-lowest firm share,99.69%,Share of stock-months with the adverse mean lo...
7,Adverse ES contribution ratio,73.78%,Adverse contribution divided by total ES


Saved: overleaf/tab_tex/12_component_numerical_checks.tex


,Diagnostic,Value,Interpretation
0,Maximum ES reconstruction error,4.44e-16,Should be close to zero
1,Maximum tail-weight sum error,6.73e-13,Should be close to zero
2,Maximum within-month weight spread,0.00e+00,Should be zero under common monthly weights
3,Mean effective component count,1.7962,Above one means more than one component is used
4,Collapsed-month share,0.93%,Share of months dominated by one component
5,Adverse-lowest month share,100.00%,Share of months with the adverse mean lowest
6,Adverse-lowest firm share,99.69%,Share of stock-months with the adverse mean lo...
7,Adverse ES contribution ratio,73.78%,Adverse contribution divided by total ES


## Table 13. Component Use and Weights by Window

This first window-level component table focuses on mixture use, collapse, label stability, and average component weights.


In [15]:
table_name = "window_components"
path = RESULT_DIR / f"{table_name}_{TIMESTAMP}.parquet"
windows = pd.read_parquet(path)


# windows = results["structured_component_windows"].copy()
test_period = (
    pd.to_datetime(windows["test_start"]).dt.strftime("%Y-%m")
    + " to "
    + pd.to_datetime(windows["test_end"]).dt.strftime("%Y-%m")
)

table_13 = pd.DataFrame({
    "Window": windows["window"],
    "Test period": test_period,
    "Months": windows["months"],
    "Effective K": windows["mean_effective_k"],
    "Collapse share (%)": 100 * windows["collapse_month_share"],
    "Adverse lowest firms (%)": 100 * windows["adverse_lowest_firm_share"],
    "Adverse weight (%)": 100 * windows["mean_pi_adverse"],
    "Normal weight (%)": 100 * windows["mean_pi_normal"],
    "Favorable weight (%)": 100 * windows["mean_pi_favorable"],
})

export_latex_table(
    table_13,
    "13_component_window_weights.tex",
    "Structured K3 Component Use by Forecast Window",
    "tab:component_window_weights",
    r"Window statistics are equal-month averages within each Test year. Effective K equals one when a single component receives all weight and increases as weight is spread across components. Collapse share is the percentage of months in which the largest weight exceeds the diagnostic threshold. Component weights are common to all firms within a month.",
    formats={
        "Window": ".0f",
        "Months": ".0f",
        "Effective K": ".3f",
        "Collapse share (%)": ".2f",
        "Adverse lowest firms (%)": ".2f",
        "Adverse weight (%)": ".2f",
        "Normal weight (%)": ".2f",
        "Favorable weight (%)": ".2f",
    },
    landscape=True,
    font_size="scriptsize",
)


,Window,Test period,Months,Effective K,Collapse share (%),Adverse lowest firms (%),Adverse weight (%),Normal weight (%),Favorable weight (%)
0,0,2007-01 to 2007-12,12,1.939,0.00,99.47,64.12,17.93,17.95
1,1,2008-01 to 2008-12,12,2.287,0.00,99.02,42.94,35.71,21.34
2,2,2009-01 to 2009-12,12,1.697,0.00,99.76,74.06,15.88,10.06
3,3,2010-01 to 2010-12,12,2.479,0.00,99.94,30.99,48.34,20.67
4,4,2011-01 to 2011-12,12,1.859,0.00,100.00,31.55,57.05,11.40
5,5,2012-01 to 2012-12,12,1.603,0.00,100.00,4.57,75.96,19.47
6,6,2013-01 to 2013-12,12,1.473,0.00,99.92,7.58,80.47,11.95
7,7,2014-01 to 2014-12,12,1.543,0.00,100.00,36.30,56.41,7.29
8,8,2015-01 to 2015-12,12,1.511,0.00,100.00,5.84,79.92,14.24
9,9,2016-01 to 2016-12,12,1.678,0.00,100.00,25.41,68.43,6.16


Saved: overleaf/tab_tex/13_component_window_weights.tex


,Window,Test period,Months,Effective K,Collapse share (%),Adverse lowest firms (%),Adverse weight (%),Normal weight (%),Favorable weight (%)
0,0,2007-01 to 2007-12,12,1.939,0.00,99.47,64.12,17.93,17.95
1,1,2008-01 to 2008-12,12,2.287,0.00,99.02,42.94,35.71,21.34
2,2,2009-01 to 2009-12,12,1.697,0.00,99.76,74.06,15.88,10.06
3,3,2010-01 to 2010-12,12,2.479,0.00,99.94,30.99,48.34,20.67
4,4,2011-01 to 2011-12,12,1.859,0.00,100.00,31.55,57.05,11.40
5,5,2012-01 to 2012-12,12,1.603,0.00,100.00,4.57,75.96,19.47
6,6,2013-01 to 2013-12,12,1.473,0.00,99.92,7.58,80.47,11.95
7,7,2014-01 to 2014-12,12,1.543,0.00,100.00,36.30,56.41,7.29
8,8,2015-01 to 2015-12,12,1.511,0.00,100.00,5.84,79.92,14.24
9,9,2016-01 to 2016-12,12,1.678,0.00,100.00,25.41,68.43,6.16


## Table 14. Component Separation and ES by Window

This second window-level table focuses on component separation and the adverse component's contribution to total ES.


In [16]:

table_14 = pd.DataFrame({
    "Window": windows["window"],
    "Test period": test_period,
    "Adverse-normal separation": windows["mean_separation_adverse_normal"],
    "Normal-favorable separation": windows["mean_separation_normal_favorable"],
    "Adverse ES contribution": windows["mean_contribution_adverse"],
    "Total ES": windows["mean_total_es"],
    "Adverse contribution share (%)": 100 * windows["mean_contribution_adverse"] / windows["mean_total_es"],
})

export_latex_table(
    table_14,
    "14_component_window_tail.tex",
    "Structured K3 Tail Decomposition by Forecast Window",
    "tab:component_window_tail",
    r"Window statistics are equal-month averages within each Test year. Separation compares adjacent component means relative to their scales. ES is reported as a positive loss. The contribution share is the adverse component's ES contribution divided by total ES.",
    formats={
        "Window": ".0f",
        "Adverse-normal separation": ".4f",
        "Normal-favorable separation": ".4f",
        "Adverse ES contribution": ".4f",
        "Total ES": ".4f",
        "Adverse contribution share (%)": ".2f",
    },
    landscape=True,
    font_size="scriptsize",
)


,Window,Test period,Adverse-normal separation,Normal-favorable separation,Adverse ES contribution,Total ES,Adverse contribution share (%)
0,0,2007-01 to 2007-12,0.9401,0.8692,0.2928,0.3312,88.41
1,1,2008-01 to 2008-12,1.0321,1.0994,0.2642,0.2944,89.71
2,2,2009-01 to 2009-12,1.0454,1.0339,0.3236,0.3270,98.95
3,3,2010-01 to 2010-12,1.2047,0.9401,0.2666,0.2973,89.68
4,4,2011-01 to 2011-12,1.3407,0.9839,0.2449,0.2871,85.29
5,5,2012-01 to 2012-12,1.3201,0.9375,0.0815,0.2054,39.68
6,6,2013-01 to 2013-12,1.1242,1.0179,0.0869,0.2041,42.56
7,7,2014-01 to 2014-12,1.0831,1.1434,0.1804,0.2408,74.89
8,8,2015-01 to 2015-12,1.1793,1.0776,0.0903,0.1967,45.88
9,9,2016-01 to 2016-12,1.3493,0.8464,0.1636,0.2074,78.88


Saved: overleaf/tab_tex/14_component_window_tail.tex


,Window,Test period,Adverse-normal separation,Normal-favorable separation,Adverse ES contribution,Total ES,Adverse contribution share (%)
0,0,2007-01 to 2007-12,0.9401,0.8692,0.2928,0.3312,88.41
1,1,2008-01 to 2008-12,1.0321,1.0994,0.2642,0.2944,89.71
2,2,2009-01 to 2009-12,1.0454,1.0339,0.3236,0.3270,98.95
3,3,2010-01 to 2010-12,1.2047,0.9401,0.2666,0.2973,89.68
4,4,2011-01 to 2011-12,1.3407,0.9839,0.2449,0.2871,85.29
5,5,2012-01 to 2012-12,1.3201,0.9375,0.0815,0.2054,39.68
6,6,2013-01 to 2013-12,1.1242,1.0179,0.0869,0.2041,42.56
7,7,2014-01 to 2014-12,1.0831,1.1434,0.1804,0.2408,74.89
8,8,2015-01 to 2015-12,1.1793,1.0776,0.0903,0.1967,45.88
9,9,2016-01 to 2016-12,1.3493,0.8464,0.1636,0.2074,78.88


## Table 15. Adverse Weight and Future Market Loss

This diagnostic asks whether the raw adverse component weight predicts the next month's equal-weighted market loss.


## Table 16. Structured K3 versus Fused K1

The original notebook also directly compares Structured K3 with Fused K1. Because this summary was not saved as its own Parquet file, it is reconstructed exactly from the saved monthly scores.


In [17]:


def newey_west_mean_test(values, lags=12):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    n = len(values)
    if n < 3:
        return np.nan, np.nan, np.nan, np.nan, n

    centered = values - values.mean()
    max_lag = min(lags, n - 1)
    long_run_variance = np.dot(centered, centered) / n
    for lag in range(1, max_lag + 1):
        autocovariance = np.dot(centered[lag:], centered[:-lag]) / n
        long_run_variance += 2 * (1 - lag / (max_lag + 1)) * autocovariance

    standard_error = np.sqrt(max(long_run_variance, 0) / n)
    t_stat = values.mean() / standard_error if standard_error > 0 else np.nan
    p_value = 2 * norm.sf(abs(t_stat)) if np.isfinite(t_stat) else np.nan
    return values.mean(), standard_error, t_stat, p_value, n


def compare_two_models(monthly_scores, candidate, benchmark, metrics, lags=12):
    left = monthly_scores[monthly_scores["model_id"] == candidate][["mthcaldt"] + metrics]
    right = monthly_scores[monthly_scores["model_id"] == benchmark][["mthcaldt"] + metrics]
    paired = left.merge(
        right,
        on="mthcaldt",
        suffixes=("_candidate", "_benchmark"),
        validate="one_to_one",
    )

    rows = []
    for metric in metrics:
        difference = paired[f"{metric}_candidate"] - paired[f"{metric}_benchmark"]
        mean_diff, nw_se, nw_t, p_value, months = newey_west_mean_test(difference, lags)
        benchmark_std = paired[f"{metric}_benchmark"].std(ddof=1)
        rows.append({
            "Metric": METRIC_LABELS[metric],
            "Mean difference": mean_diff,
            "95% CI lower": mean_diff - 1.96 * nw_se,
            "95% CI upper": mean_diff + 1.96 * nw_se,
            "Standardized difference": mean_diff / benchmark_std,
            "NW t": nw_t,
            "p-value": p_value,
            "Months": months,
        })
    return pd.DataFrame(rows)

table_name = "monthly_scores"
path = RESULT_DIR / f"{table_name}_{TIMESTAMP}.parquet"
summary = pd.read_parquet(path)

table_16 = compare_two_models(
    pd.read_parquet(path),
    candidate="structured_normal_k3",
    benchmark="fused_normal_k1",
    metrics=["fz0_5", "tail_crps_10", "nll", "crps", "qs_1", "qs_5", "qs_10"],
    lags=12,
)

export_latex_table(
    table_16,
    "16_structured_vs_fused_k1.tex",
    "Structured K3 versus Fused K1",
    "tab:structured_vs_fused_k1",
    r"Each difference is Structured K3 monthly loss minus Fused K1 monthly loss, so a negative value favors Structured K3. Confidence intervals and t-statistics use Newey--West standard errors with 12 lags. The standardized difference divides the mean difference by the time-series standard deviation of the Fused K1 monthly loss.",
    formats={
        "Mean difference": ".6f",
        "95% CI lower": ".6f",
        "95% CI upper": ".6f",
        "Standardized difference": ".4f",
        "NW t": ".3f",
        "p-value": ".4f",
        "Months": ".0f",
    },
    landscape=True,
    font_size="scriptsize",
)


,Metric,Mean difference,95% CI lower,95% CI upper,Standardized difference,NW t,p-value,Months
0,FZ0 (5%),-0.054496,-0.121090,0.012097,-0.0593,-1.604,0.1087,216
1,Tail CRPS,-0.000827,-0.001669,0.000016,-0.0757,-1.923,0.0545,216
2,NLL,-0.147770,-0.209501,-0.086039,-0.2239,-4.692,0.0000,216
3,CRPS,-0.003450,-0.005265,-0.001635,-0.1340,-3.725,0.0002,216
4,QS (1%),-0.000238,-0.000634,0.000158,-0.0430,-1.178,0.2386,216
5,QS (5%),-0.000757,-0.001670,0.000157,-0.0647,-1.624,0.1044,216
6,QS (10%),-0.001719,-0.003158,-0.000281,-0.1018,-2.343,0.0192,216


Saved: overleaf/tab_tex/16_structured_vs_fused_k1.tex


,Metric,Mean difference,95% CI lower,95% CI upper,Standardized difference,NW t,p-value,Months
0,FZ0 (5%),-0.054496,-0.121090,0.012097,-0.0593,-1.604,0.1087,216
1,Tail CRPS,-0.000827,-0.001669,0.000016,-0.0757,-1.923,0.0545,216
2,NLL,-0.147770,-0.209501,-0.086039,-0.2239,-4.692,0.0000,216
3,CRPS,-0.003450,-0.005265,-0.001635,-0.1340,-3.725,0.0002,216
4,QS (1%),-0.000238,-0.000634,0.000158,-0.0430,-1.178,0.2386,216
5,QS (5%),-0.000757,-0.001670,0.000157,-0.0647,-1.624,0.1044,216
6,QS (10%),-0.001719,-0.003158,-0.000281,-0.1018,-2.343,0.0192,216


# Checks and Handoff

The next cells verify that every table was written, that labels are unique, and that no unresolved `NaN` or infinite values entered the LaTeX files.


## Compile all tables

This final smoke test creates a small validation document and compiles every generated table. A successful run confirms that the fragments are valid LaTeX with the same core table packages used by the paper.


## How to use a table in `main.tex`

Insert a generated table with one line, for example:

```latex
\input{tables/model_comparison/03_primary_forecast_performance.tex}
```

When model results change:

1. rerun `3Compare_model.ipynb` and save the new Parquet results;
2. update `TIMESTAMP` near the top of this notebook;
3. run this notebook from top to bottom;
4. recompile `main.tex`.

No table numbers are hard-coded. LaTeX assigns the numbers from the order in which the `\input{}` commands appear in the paper.
